# HalluScope — Replication runs on Colab (free T4)

Runs the three replication models of Joo et al. (2025) — LLaMA-3-8B-Instruct, Qwen2-7B-Instruct, Gemma-2-9B-it — over the 297 replication prompts with greedy decoding and raw-logit entropy capture, in 4-bit NF4.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**.
2. Accept the licenses on the HF pages of `meta-llama/Meta-Llama-3-8B-Instruct` and `google/gemma-2-9b-it` with your HF account.
3. Add your HF token as a Colab **secret** named `HF_TOKEN` (key icon in the left sidebar) and allow notebook access.

Each model takes roughly **1.5–3 h**. Generation appends + flushes per prompt and **resumes automatically** — if the runtime disconnects, just reconnect and re-run the same cell.

In [ ]:
!nvidia-smi

In [ ]:
%pip install -q "transformers>=4.38,<5" accelerate bitsandbytes spacy packaging pandas scikit-learn matplotlib
!python -m spacy download en_core_web_sm -q

In [ ]:
# HF auth for the gated models (Llama-3, Gemma-2)
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("HF token set:", bool(os.environ["HF_TOKEN"]))

In [ ]:
# Get the code. If the repo is private, upload a zip of the HalluScope
# folder instead (Files sidebar) and unzip it here.
import os
REPO_URL = "https://github.com/SherwanAli0/HalluScope.git"
if not os.path.exists("HalluScope"):
    !git clone $REPO_URL
%cd HalluScope

## Model 1/3 — LLaMA-3-8B-Instruct (gated)

In [ ]:
!python integration/run_pipeline.py --model meta-llama/Meta-Llama-3-8B-Instruct --alias llama3-8b-instruct --load-4bit

## Model 2/3 — Qwen2-7B-Instruct (open)

In [ ]:
!python integration/run_pipeline.py --model Qwen/Qwen2-7B-Instruct --alias qwen2-7b-instruct --load-4bit

## Model 3/3 — Gemma-2-9B-it (gated; eager attention is forced by the script)

In [ ]:
!python integration/run_pipeline.py --model google/gemma-2-9b-it --alias gemma2-9b-it --load-4bit

## Aggregate + save everything to Drive

Builds the replication Table I and classification metrics from whatever aliases finished, then zips all stage outputs to your Drive. Download `halluscope_outputs.zip`, unzip it into the repo locally, and you have the full results.

In [ ]:
import os
aliases = [a for a in ["llama3-8b-instruct", "qwen2-7b-instruct", "gemma2-9b-it"]
           if os.path.exists(f"evaluation/outputs/{a}")]
print("aggregating:", aliases)
if aliases:
    !python integration/run_pipeline.py --aggregate {" ".join(aliases)}

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!zip -qr /content/drive/MyDrive/halluscope_outputs.zip generation/outputs entropy/outputs evaluation/outputs results
print("saved: MyDrive/halluscope_outputs.zip")